<div align="center">

**Ejemplo Práctico: Predicción USD/CLP con Modelos Lineales Regularizados**

</div>

# ¿Por qué hacer Regresión? El Problema del Tipo de Cambio

## Contexto del problema

El **tipo de cambio USD/CLP** es una variable crítica en finanzas: afecta importaciones, exportaciones, valoración de activos y política monetaria. Queremos **predecir** su valor futuro usando variables macroeconómicas y financieras disponibles.

### ¿Por qué regresión lineal?

1. **Objetivo cuantitativo**: La variable objetivo (USD/CLP) es **continua** — no clasificamos categorías, sino que estimamos un valor numérico.
2. **Relación lineal aproximada**: Muchos factores (tasas de interés, commodities, volatilidad) tienen efectos **aditivos** sobre el tipo de cambio. Un modelo lineal captura estas relaciones de forma interpretable.
3. **Interpretabilidad**: Los coeficientes $\beta$ indican **cuánto cambia** el tipo de cambio (en CLP) por cada unidad de cambio en cada predictor — útil para análisis de riesgo y toma de decisiones.

### ¿Por qué regresión regularizada?

1. **Muchos predictores**: Tenemos tasas de interés (USSW1, USSW10), swaps (CHSWP), crédito (CCHIL), commodities (Cobre, Oro) y volatilidad (VIX). Con pocas observaciones vs. muchas variables, la regresión lineal puede **sobreajustar**.
2. **Multicolinealidad**: Tasas de interés de distintos plazos están **correlacionadas**. Con Ridge/Lasso reducimos inestabilidad en los coeficientes.
3. **Selección de características**: Lasso identifica qué predictores son **realmente importantes** para el tipo de cambio.
4. **Generalización**: Queremos que el modelo funcione en datos **no vistos** (testing), no solo en el entrenamiento.

## Carga de datos y exploración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Cargar datos (requiere openpyxl: pip install openpyxl; o usar data del proyecto)
data = pd.read_excel('data_usd-clp.xlsx')
print("Dimensiones:", data.shape)
print("\nColumnas:")
print(data.columns.tolist())
data.head(10)

### Variables en el dataset

| Variable | Descripción | Rol en el modelo |
|----------|-------------|------------------|
| **USDCLP Curncy** | Tipo de cambio USD/CLP (spot) | **Variable objetivo (y)** |
| **USSW1 Curncy** | Tasa swap USD 1 año | Predictor |
| **USSW10 Curncy** | Tasa swap USD 10 años | Predictor |
| **CHSWP1** / **CHSWP10** | Swaps Chile 1/10 años | Predictor |
| **CCHIL1U5 CBIN** | Credit swap Chile 5 años | Predictor |
| **HGA Comdty** | Oro (oro) | Predictor |
| **CL1 Comdty** | Petróleo WTI | Predictor |
| **VIX Index** | Índice de volatilidad | Predictor |

**Interpretación**: Tasas de interés, spreads de crédito y commodities son factores que influyen en el tipo de cambio de un país emergente como Chile.

In [ ]:
# Preparar datos: índice temporal y variables y, X
data_proc = data.set_index(data.columns[0])
data_proc.index = pd.to_datetime(data_proc.index)

y = data_proc.iloc[:, 0]   # USD/CLP
X = data_proc.iloc[:, 1:-1]  # Predictores (excluye última columna si es redundante)

print("Variable objetivo (y):", y.name)
print("Predictores:", list(X.columns))
print("\nEstadísticas descriptivas:")
print(X.describe().round(4))

In [ ]:
# Visualización de la serie USD/CLP
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(y.index, y.values, color='#2b8cbe', linewidth=1)
ax.set_title('Tipo de cambio USD/CLP (serie histórica)')
ax.set_xlabel('Fecha')
ax.set_ylabel('CLP por USD')
plt.tight_layout()
plt.show()

## División Training / Testing

Separamos por **fecha**: entrenamos con datos hasta junio 2016 y evaluamos con datos posteriores. Esto simula un escenario real: predecir el futuro con información pasada.

In [ ]:
fecha_corte = pd.to_datetime('2016-06-19')

X_train = X[X.index < fecha_corte].copy()
y_train = y[y.index < fecha_corte].copy()
X_test  = X[X.index >= fecha_corte].copy()
y_test  = y[y.index >= fecha_corte].copy()

print(f"Training: {len(X_train)} obs. ({X_train.index.min().date()} a {X_train.index.max().date()})")
print(f"Testing:  {len(X_test)} obs. ({X_test.index.min().date()} a {X_test.index.max().date()})")
print(f"\nNúmero de predictores: {X_train.shape[1]}")

## Estandarización

**Por qué estandarizar**: Las variables tienen escalas muy distintas (tasas en %, VIX en puntos, commodities en USD). La regularización penaliza **magnitudes** de coeficientes; sin estandarizar, los predictores con escala mayor dominarían artificialmente.

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print("Datos estandarizados (media≈0, std≈1)")
print(pd.DataFrame(X_train_sc, columns=X_train.columns).describe().round(4))

# Estimación de modelos

## Linear, Ridge, Lasso, Elastic Net

Estimamos los cuatro modelos con hiperparámetros por defecto (alpha=1 en Ridge/Lasso/ElasticNet).

In [ ]:
lr = LinearRegression()
ridge = Ridge()
lasso = Lasso()
elastic = ElasticNet()

lr.fit(X_train_sc, y_train)
ridge.fit(X_train_sc, y_train)
lasso.fit(X_train_sc, y_train)
elastic.fit(X_train_sc, y_train)

print("Modelos ajustados correctamente.")

## Tabla de coeficientes: interpretación detallada

Cada fila es un predictor; cada columna es un modelo. **Interpretación de los coeficientes** (con datos estandarizados):

- Un coeficiente de +10 significa: si el predictor aumenta 1 desviación estándar, el USD/CLP aumenta en promedio ~10 CLP.
- Un coeficiente de 0 (Lasso/Elastic Net) significa: esa variable fue **eliminada** del modelo (no se considera relevante).
- Coeficientes más pequeños en Ridge vs. Linear: Ridge **contrae** los coeficientes hacia cero.

In [ ]:
coef = pd.DataFrame({
    'Linear': lr.coef_,
    'Ridge': ridge.coef_,
    'Lasso': lasso.coef_,
    'ElasticNet': elastic.coef_
}, index=X_train.columns)

print("Coeficientes (β) por modelo:")
display(coef.round(4))

print("\n--- Interpretación rápida ---")
print(f"Lasso: {np.sum(lasso.coef_ != 0)} de {len(lasso.coef_)} predictores con coef ≠ 0 (selección)")
print(f"Elastic Net: {np.sum(elastic.coef_ != 0)} de {len(elastic.coef_)} predictores con coef ≠ 0 (selección)")
print(f"Ridge: todos los predictores con coef ≠ 0 (solo contracción)")

### ¿Por qué los coeficientes difieren entre modelos?

| Modelo | Comportamiento | Implicación práctica |
|--------|----------------|----------------------|
| **Linear** | Minimiza solo RSS | Puede sobreajustar; coeficientes inestables si hay correlación |
| **Ridge** | Penaliza $\sum \beta^2$ | Contrae todos; ninguno llega a cero |
| **Lasso** | Penaliza $\sum |\beta|$ | Algunos coeficientes = 0 → selección automática |
| **Elastic Net** | Combinación L1+L2 | Equilibrio entre contracción y selección |

Si Lasso pone 0 en **USSW1** o **USSW10**, sugiere que esa variable no aporta información adicional al modelo (bien por correlación con otras o por falta de poder predictivo).

## Métricas de desempeño

Evaluamos en **Training** y **Testing** para detectar sobreajuste:
- **MAE** (Mean Absolute Error): error promedio en CLP.
- **RMSE** (Root Mean Squared Error): penaliza más los errores grandes.
- **$R^2$**: proporción de varianza explicada (1 = perfecto).

**Regla clave**: Si el error en Testing es *mucho mayor* que en Training → **sobreajuste**.

In [ ]:
def metricas(y_real, y_pred, nombre):
    return {
        'Modelo': nombre,
        'MAE': mean_absolute_error(y_real, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_real, y_pred)),
        'R2': r2_score(y_real, y_pred)
    }

modelos = [('Linear', lr), ('Ridge', ridge), ('Lasso', lasso), ('ElasticNet', elastic)]

res_train = []
res_test = []

for nombre, modelo in modelos:
    pred_train = modelo.predict(X_train_sc)
    pred_test  = modelo.predict(X_test_sc)
    res_train.append(metricas(y_train, pred_train, nombre))
    res_test.append(metricas(y_test, pred_test, nombre))

df_train = pd.DataFrame(res_train)
df_test = pd.DataFrame(res_test)

print("Métricas en TRAINING:")
display(df_train.round(6))
print("\nMétricas en TESTING:")
display(df_test.round(6))

## Interpretación de las métricas

1. **Linear Regression en Training**: MAE/RMSE casi cero y $R^2 \approx 1$ indica **ajuste perfecto** en training — pero con muchos predictores y pocas restricciones, es señal de **sobreajuste**.

2. **Ridge, Lasso, Elastic Net**: tienen error en training mayor que Linear, pero eso es **intencional** — sacrifican algo de ajuste en training por mejor **generalización**.

3. **Comparar Training vs Testing**: Si el error en Testing es similar o menor que en Training para los modelos regularizados, están generalizando mejor que Linear.

4. **MAE en CLP**: Un MAE de 5 en testing significa que en promedio nos equivocamos en ~5 CLP por dólar.

In [ ]:
# Resumen comparativo
comparativo = pd.merge(
    df_train.add_suffix('_train'),
    df_test.add_suffix('_test'),
    left_on='Modelo_train',
    right_on='Modelo_test'
)[['Modelo_train', 'MAE_train', 'MAE_test', 'RMSE_train', 'RMSE_test', 'R2_train', 'R2_test']]
comparativo.columns = ['Modelo', 'MAE_tr', 'MAE_tst', 'RMSE_tr', 'RMSE_tst', 'R2_tr', 'R2_tst']
print("Comparativo Training vs Testing:")
display(comparativo.round(6))

## Resumen: ¿Cuándo usar cada método?

| Situación | Método recomendado | Razón |
|-----------|--------------------|-------|
| Pocos predictores, sin correlación | Linear | Simplicidad |
| Muchos predictores correlacionados | Ridge | Reduce varianza sin eliminar variables |
| Quieres saber qué variables importan | Lasso | Selección automática (coef ≠ 0) |
| Alta correlación + interés en selección | Elastic Net | Combina Ridge + Lasso |

**Para USD/CLP**: Con tasas, swaps, commodities y VIX correlacionados, Ridge/Lasso/Elastic Net son más robustos que Linear puro.

## Visualización: predicciones vs reales (Testing)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, (nombre, modelo) in enumerate(modelos):
    pred = modelo.predict(X_test_sc)
    axes[i].scatter(y_test, pred, alpha=0.6, s=20)
    axes[i].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2]
    axes[i].set_title(f'{nombre}')
    axes[i].set_xlabel('USD/CLP real')
    axes[i].set_ylabel('USD/CLP predicho')
    axes[i].set_aspect('equal')

plt.suptitle('Predicciones vs valores reales (muestra Testing)')
plt.tight_layout()
plt.show()